In [ ]:
import torch
import torch.nn as nn
import sklearn

from sklearn.model_selection import train_test_split
from sklearn.datasets import make_circles

In [ ]:
# Create dataset

n_samples = 1000

X, y = make_circles(
    n_samples,
    noise=0.03,
    random_state=42
)

In [ ]:
# Turn data to tensors
X = torch.from_numpy(X).type(torch.float)
y = torch.from_numpy(y).type(torch.float)

In [ ]:
# Split data to train set and test set
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [ ]:
# Set up device and random seed
torch.manual_seed(42)
torch.cuda.manual_seed(42)
device = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
# Set up two models
model_linear = nn.Sequential(
    nn.Linear(in_features=2, out_features=128),
    nn.Linear(in_features=128, out_features=256),
    nn.Linear(in_features=256, out_features=1024),
    nn.Linear(in_features=1024, out_features=1)
).to(device)

model_non_linear = nn.Sequential(
    nn.Linear(in_features=2, out_features=128),
    nn.ReLU(),
    nn.Linear(in_features=128, out_features=256),
    nn.ReLU(),
    nn.Linear(in_features=256, out_features=1024),
    nn.Linear(in_features=1024, out_features=1)
).to(device)

model_non_linear_small = nn.Sequential(
    nn.Linear(in_features=2, out_features=4),
    nn.ReLU(),
    nn.Linear(in_features=4, out_features=8),
    nn.ReLU(),
    nn.Linear(in_features=8, out_features=16),
    nn.Linear(in_features=16, out_features=1)
).to(device)

# Put two models in a list
models = [model_linear, model_non_linear, model_non_linear_small]

In [ ]:
# Training

## Accuracy
def accuracy_fn(y_true, y_pred):
  correct = torch.eq(y_true, y_pred).sum().item()
  acc = (correct/len(y_pred)) * 100
  return acc

# Epoch
epochs = 100

## Training loop
X_train, y_train = X_train.to(device), y_train.to(device)
X_test, y_test = X_test.to(device), y_test.to(device)

for model in models:

  ## loss
  loss_fn = nn.BCEWithLogitsLoss()

  ## optimizer must be put inside the loop
  optimizer = torch.optim.Adam(
    params=model.parameters(),
    lr=0.01
)

  for epoch in range(epochs):
    model.train()

    y_logits = model(X_train).squeeze()
    y_pred = torch.round(torch.sigmoid(y_logits))

    loss = loss_fn(y_logits, y_train)
    acc = accuracy_fn(y_true=y_train, y_pred=y_pred)

    optimizer.zero_grad()

    loss.backward()

    optimizer.step()

    # Testing
    model.eval()
    with torch.inference_mode():
      test_logits = model(X_test).squeeze()
      test_pred = torch.round(torch.sigmoid(test_logits))

      test_loss = loss_fn(test_logits, y_test)
      test_acc = accuracy_fn(y_true=y_test, y_pred=test_pred)

    if epoch % 10 == 0:
      print(f"Epoch: {epoch} | Loss: {loss:.5f}, Accuracy: {acc:.2f}% | Test loss: {test_loss:.5f}, Test acc: {test_acc:.2f}%")








Epoch: 0 | Loss: 0.69486, Accuracy: 52.25% | Test loss: 3.35907, Test acc: 50.00%
Epoch: 10 | Loss: 1.49802, Accuracy: 50.00% | Test loss: 0.99621, Test acc: 50.00%
Epoch: 20 | Loss: 0.70973, Accuracy: 50.00% | Test loss: 0.76983, Test acc: 50.00%
Epoch: 30 | Loss: 0.74570, Accuracy: 50.00% | Test loss: 0.70667, Test acc: 50.00%
Epoch: 40 | Loss: 0.69379, Accuracy: 50.25% | Test loss: 0.69846, Test acc: 42.50%
Epoch: 50 | Loss: 0.69334, Accuracy: 50.00% | Test loss: 0.69392, Test acc: 46.50%
Epoch: 60 | Loss: 0.69492, Accuracy: 50.00% | Test loss: 0.69674, Test acc: 50.00%
Epoch: 70 | Loss: 0.69299, Accuracy: 50.88% | Test loss: 0.69494, Test acc: 45.00%
Epoch: 80 | Loss: 0.69320, Accuracy: 43.12% | Test loss: 0.69443, Test acc: 42.50%
Epoch: 90 | Loss: 0.69300, Accuracy: 50.75% | Test loss: 0.69488, Test acc: 45.50%
Epoch: 0 | Loss: 0.69403, Accuracy: 46.62% | Test loss: 0.92631, Test acc: 50.00%
Epoch: 10 | Loss: 0.72852, Accuracy: 50.00% | Test loss: 0.75536, Test acc: 50.00%
Epoch: